## RQ 7 – final models tested on a different Dataset  

Train four CNN variants on Nickparvar and test on Nickparvar and BraTS 2023.

In [ ]:
import os, sys, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import torchvision.transforms as T, torchvision.models as models
import cv2
from PIL import Image
import torchvision.models as models

PROJECT_ROOT = Path.cwd().parent
sys.path.append(PROJECT_ROOT.as_posix())

from src.dataset import BrainTumorDataset
from src.models import SimpleCNN
from src.lpconv import LpCNN, init_lp_weights

random.seed(0); np.random.seed(0); torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', DEVICE)

In [ ]:
IMG_SIZE = 224
def clahe_pil(img):
    arr = np.array(img, dtype=np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return Image.fromarray(clahe.apply(arr))
def build_preprocess():
    return T.Compose([T.Lambda(clahe_pil),
                      T.Resize((IMG_SIZE, IMG_SIZE)),
                      T.ToTensor(),
                      T.Lambda(lambda x: (x - x.mean())/(x.std()+1e-7))])
BASE = build_preprocess()
TRAIN_TF = T.Compose([T.RandomHorizontalFlip(), *BASE.transforms])
VAL_TF = BASE

In [ ]:
BATCH = 16
NICK_ROOT = PROJECT_ROOT / 'data' / 'archive'
def loaders():
    tr = BrainTumorDataset(NICK_ROOT,'Training',transform=TRAIN_TF)
    va = BrainTumorDataset(NICK_ROOT,'Testing', transform=VAL_TF)
    return (DataLoader(tr,BATCH,shuffle=True,num_workers=4,pin_memory=True),
            DataLoader(va,BATCH,shuffle=False,num_workers=4,pin_memory=True))
train_ld, val_ld = loaders()

In [ ]:
class DoubleHeadCNN(nn.Module):
    def __init__(self, backbone, feat):
        super().__init__()
        self.backbone = backbone
        self.head_bin = nn.Linear(feat,2)
        self.head_tri = nn.Linear(feat,3)
    def forward(self,x):
        f = self.backbone(x)
        b = self.head_bin(f)
        t = self.head_tri(f)
        tp = torch.softmax(t,1)
        return torch.cat([b[:,:1], b[:,1:2]*tp],1)

In [ ]:
def train_ep(m,ld,crit,opt):
    m.train()
    for x,y in ld:
        x,y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); crit(m(x),y).backward(); opt.step()
@torch.no_grad()
def evaluate(m,ld,crit):
    m.eval(); corr=tot=0; yt=[]; yp=[]
    for x,y in ld:
        x,y = x.to(DEVICE), y.to(DEVICE)
        o=m(x); corr+=(o.argmax(1)==y).sum().item(); tot+=y.size(0)
        yt+=y.cpu().tolist(); yp+=o.argmax(1).cpu().tolist()
    return corr/tot, f1_score(yt,yp,average='macro')

In [ ]:
EPOCHS=10; LR=1e-3; WD=1e-4; crit=nn.CrossEntropyLoss()
res=[]

In [ ]:
from collections import defaultdict
history = defaultdict(list)

def train_one_epoch(model, name, crit, ld, opt):
    model.train()
    running_loss, corr, tot = 0.0, 0, 0
    for x, y in ld:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out = model(x)
        loss = crit(out, y)
        loss.backward()
        opt.step()
        running_loss += loss.item() * x.size(0)
        corr += (out.argmax(1) == y).sum().item()
        tot += x.size(0)
    history[(name, 'train_loss')].append(running_loss / tot)
    history[(name, 'train_acc')].append(corr / tot)


In [ ]:
def run(name, model):
    model.to(DEVICE)
    optimiser = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    best = 0.0
    for ep in range(EPOCHS):
        train_one_epoch(model, name, crit, train_ld, optimiser)
        val_acc, val_f1 = evaluate(model, val_ld, crit)
        history[(name, 'val_acc')].append(val_acc)
        history[(name, 'val_f1')].append(val_f1)
        best = max(best, val_acc)
    res.append({'model': name, 'nick_acc': best, 'net': model})


In [ ]:
run('SCNN-4', SimpleCNN(num_classes=4, input_size=IMG_SIZE))

C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# Cell: RN18‑TL
train_ld = DataLoader(train_ld.dataset, BATCH, shuffle=True,
                      num_workers=0, pin_memory=True)
val_ld   = DataLoader(val_ld.dataset,   BATCH, shuffle=False,
                      num_workers=0, pin_memory=True)
rn18 = models.resnet18(weights='IMAGENET1K_V1')
old_weight = rn18.conv1.weight.data
rn18.conv1 = nn.Conv2d(in_channels=1,
                       out_channels=64,
                       kernel_size=7,
                       stride=2,
                       padding=3,
                       bias=False)
with torch.no_grad():
    rn18.conv1.weight[:] = old_weight.mean(dim=1, keepdim=True)

rn18.fc = nn.Linear(rn18.fc.in_features, 4)

run('RN18-TL', rn18)


In [ ]:
# Cell: run LpCNN with 1‑channel input  ✅
def convert_first_conv_to_grayscale(model):
    """
    Find the first Conv2d with in_channels == 3, replace it with
    an identical layer that has in_channels == 1, and copy weights
    by averaging the RGB kernels.
    """
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d) and module.in_channels == 3:
            # Build the new layer
            new_conv = nn.Conv2d(
                in_channels=1,
                out_channels=module.out_channels,
                kernel_size=module.kernel_size,
                stride=module.stride,
                padding=module.padding,
                dilation=module.dilation,
                groups=module.groups,
                bias=(module.bias is not None),
                padding_mode=module.padding_mode,
            )
            # Copy weights (average over channel dim)
            with torch.no_grad():
                new_conv.weight.copy_(module.weight.mean(dim=1, keepdim=True))
                if module.bias is not None:
                    new_conv.bias.copy_(module.bias)
            # Replace the layer inside its parent
            parent = model
            path = name.split(".")
            for p in path[:-1]:
                parent = getattr(parent, p)
            setattr(parent, path[-1], new_conv)
            break
    return model


lp = LpCNN(num_classes=4, in_channels=1)
lp.apply(init_lp_weights)
run("LpCNN", lp)


In [ ]:
back=SimpleCNN(num_classes=4, input_size=IMG_SIZE).features
# Calculate the correct feature dimension size
with torch.no_grad():
	dummy = torch.zeros(1, 1, IMG_SIZE, IMG_SIZE)
	out = back(dummy)
	fd = out.shape[1] * out.shape[2] * out.shape[3]  # Calculate actual flattened dimensions

dh=DoubleHeadCNN(nn.Sequential(back, nn.Flatten()), fd)
run('DH-CNN', dh)
pd.DataFrame([{k:v for k,v in d.items() if k!='net'} for d in res])

In [ ]:
# plot accuracy curves
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
for entry in res:
    name = entry['model']
    plt.plot(history[(name, 'val_acc')], label=name)
plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.title('Nickparvar validation curves')
plt.legend(); plt.grid(True); plt.show()


In [ ]:
# confusion matrix for the best Nickparvar model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best_entry = max(res, key=lambda d: d['nick_acc'])
best_model = best_entry['net'].eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_ld:
        x = x.to(DEVICE)
        y_true.extend(y.numpy())
        y_pred.extend(best_model(x).argmax(1).cpu().numpy())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=val_ld.dataset.classes)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title(f'Confusion matrix – {best_entry["model"]}')
plt.show()


In [ ]:
BRATS_ROOT = PROJECT_ROOT/'data'/'brats_slices'
brats_ld = DataLoader(BrainTumorDataset(BRATS_ROOT,'glioma',transform=VAL_TF,single_class=True),
                      BATCH, shuffle=False, num_workers=4, pin_memory=True)
for d in res:
    acc,f1=evaluate(d['net'], brats_ld, crit)
    d['brats_acc']=acc; d['brats_f1']=f1
final=pd.DataFrame([{k:v for k,v in d.items() if k!='net'} for d in res])
final